# UrbanSound8K Audio Classification with a CNN

This notebook builds an end-to-end audio classification pipeline. It explores the UrbanSound8K metadata, loads and preprocesses audio, converts recordings into Mel spectrograms, trains a convolutional neural network (CNN), and evaluates its performance on a held-out test set.

**Important:** the dataset contains 8,732 labelled audio clips across 10 sound classes. The notebook processes the full dataset by default, so feature extraction can take some time. A smaller `MAX_SAMPLES` value can be used for a quick test run.

In [ ]:
# Import the libraries used throughout the notebook
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import kagglehub
import tensorflow as tf

from IPython.display import Audio, display
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Download UrbanSound8K from Kaggle through kagglehub.
# On the first run, Kaggle access may need to be configured in the Colab runtime.
dataset_path = kagglehub.dataset_download("chrisfilo/urbansound8k")
print("Dataset downloaded to:", dataset_path)

metadata_path = os.path.join(dataset_path, "UrbanSound8K.csv")
if not os.path.exists(metadata_path):
    raise FileNotFoundError(f"Could not find UrbanSound8K.csv under {dataset_path}")

In [ ]:
# Read the metadata file. Each row describes one labelled audio clip.
metadata = pd.read_csv(metadata_path)
print("Dataset shape:", metadata.shape)
display(metadata.head())
display(metadata.info())

In [ ]:
# Count the recordings and inspect the number of examples in each sound class.
print("Total audio records:", len(metadata))
class_counts = metadata["class"].value_counts().sort_index()
display(class_counts.to_frame("number_of_audio_files"))

plt.figure(figsize=(12, 5))
class_counts.plot(kind="bar")
plt.title("Number of audio recordings per class")
plt.xlabel("Sound class")
plt.ylabel("Number of recordings")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# UrbanSound8K normally stores recordings in audio/fold1 ... audio/fold10.
# This resolver also supports dataset copies where the fold folders are at the root.
audio_root = os.path.join(dataset_path, "audio")
if not os.path.isdir(audio_root):
    audio_root = dataset_path

def get_audio_path(row):
    fold = f"fold{int(row['fold'])}"
    filename = row["slice_file_name"]
    candidate = os.path.join(audio_root, fold, filename)
    if not os.path.isfile(candidate):
        raise FileNotFoundError(f"Audio file not found: {candidate}")
    return candidate

metadata["audio_path"] = metadata.apply(get_audio_path, axis=1)
print("Example audio path:", metadata.loc[0, "audio_path"])
display(metadata[["slice_file_name", "class", "audio_path"]].head())

In [ ]:
# Load audio as mono at a consistent sample rate and make every clip four seconds long.
def load_audio_file(file_path, target_sample_rate=22050, duration=4):
    audio, sr = librosa.load(
        file_path,
        sr=target_sample_rate,
        mono=True,
        duration=duration
    )

    target_length = target_sample_rate * duration
    if len(audio) < target_length:
        audio = np.pad(audio, (0, target_length - len(audio)))
    else:
        audio = audio[:target_length]

    return audio.astype(np.float32), sr

In [ ]:
# Load and plot the first recording to inspect its waveform.
first_audio_path = metadata.loc[0, "audio_path"]
first_audio, sample_rate = load_audio_file(first_audio_path)

plt.figure(figsize=(14, 4))
librosa.display.waveshow(first_audio, sr=sample_rate)
plt.title(f"Waveform: {metadata.loc[0, 'class']}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

print("Audio duration:", len(first_audio) / sample_rate, "seconds")
print("Sample rate:", sample_rate)

In [ ]:
# Listen to the first four-second recording.
display(Audio(first_audio, rate=sample_rate))

In [ ]:
# Remove leading and trailing silence using librosa's energy-based trimming.
def trim_audio(audio, top_db=20):
    trimmed_audio, index = librosa.effects.trim(audio, top_db=top_db)
    return trimmed_audio, index

trimmed_audio, trim_indices = trim_audio(first_audio)
print("Original samples:", len(first_audio))
print("Trimmed samples:", len(trimmed_audio))

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
librosa.display.waveshow(first_audio, sr=sample_rate, ax=axes[0])
axes[0].set_title("Original audio")
librosa.display.waveshow(trimmed_audio, sr=sample_rate, ax=axes[1])
axes[1].set_title("Audio after silence trimming")
plt.tight_layout()
plt.show()

In [ ]:
# Add low-level random noise. This is useful for demonstrating augmentation
# and checking whether the audio representation is sensitive to background noise.
def add_random_noise(audio, noise_factor=0.05):
    noise = np.random.randn(len(audio)).astype(np.float32)
    noisy_audio = audio + noise_factor * noise
    return np.clip(noisy_audio, -1.0, 1.0)

noisy_audio = add_random_noise(first_audio)
display(Audio(noisy_audio, rate=sample_rate))

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
librosa.display.waveshow(first_audio, sr=sample_rate, ax=axes[0])
axes[0].set_title("Original audio")
librosa.display.waveshow(noisy_audio, sr=sample_rate, ax=axes[1])
axes[1].set_title("Audio with added random noise")
plt.tight_layout()
plt.show()

In [ ]:
# Convert a waveform into a log-scaled Mel spectrogram.
def audio_to_mel_spectrogram(audio, sample_rate=22050, n_mels=128, hop_length=512):
    mel_spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sample_rate,
        n_mels=n_mels,
        hop_length=hop_length,
        n_fft=2048,
        power=2.0
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    return mel_spec_db.astype(np.float32)

first_mel = audio_to_mel_spectrogram(trimmed_audio, sample_rate)
plt.figure(figsize=(12, 5))
librosa.display.specshow(
    first_mel, sr=sample_rate, hop_length=512,
    x_axis="time", y_axis="mel"
)
plt.colorbar(format="%+2.0f dB")
plt.title(f"Mel spectrogram: {metadata.loc[0, 'class']}")
plt.tight_layout()
plt.show()

# Compare one example from several classes, when those classes are present.
classes_to_compare = ["gun_shot", "children_playing", "dog_bark"]
fig, axes = plt.subplots(len(classes_to_compare), 1, figsize=(12, 10))
for ax, class_name in zip(axes, classes_to_compare):
    match = metadata[metadata["class"] == class_name]
    if match.empty:
        ax.set_title(f"{class_name}: class not found")
        continue
    audio, sr = load_audio_file(match.iloc[0]["audio_path"])
    mel = audio_to_mel_spectrogram(audio, sr)
    librosa.display.specshow(mel, sr=sr, hop_length=512, x_axis="time", y_axis="mel", ax=ax)
    ax.set_title(class_name)
plt.tight_layout()
plt.show()

In [ ]:
# Convert each recording into a fixed-size Mel spectrogram for CNN input.
# MAX_SAMPLES=None processes all 8,732 recordings. Set it to an integer for a quick smoke test.
MAX_SAMPLES = None
SAMPLE_RATE = 22050
MAX_PAD_LEN = 173

def prepare_cnn_data(dataframe, sample_rate=SAMPLE_RATE, max_pad_len=MAX_PAD_LEN):
    X, y = [], []
    working_data = dataframe if MAX_SAMPLES is None else dataframe.groupby("class", group_keys=False).head(
        max(1, MAX_SAMPLES // dataframe["class"].nunique())
    )

    for row in working_data.itertuples(index=False):
        audio, _ = load_audio_file(row.audio_path, target_sample_rate=sample_rate, duration=4)
        trimmed_audio, _ = trim_audio(audio)
        mel_spec = audio_to_mel_spectrogram(trimmed_audio, sample_rate)

        if mel_spec.shape[1] < max_pad_len:
            pad_width = max_pad_len - mel_spec.shape[1]
            mel_spec = np.pad(mel_spec, ((0, 0), (0, pad_width)), mode="constant")
        else:
            mel_spec = mel_spec[:, :max_pad_len]

        X.append(mel_spec)
        y.append(int(row.classID))

    X = np.asarray(X, dtype=np.float32)[..., np.newaxis]
    y = np.asarray(y, dtype=np.int64)
    return X, y

full_metadata = metadata.reset_index(drop=True)
X, y = prepare_cnn_data(full_metadata)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of examples:", len(y))
print("Number of classes:", len(np.unique(y)))

In [ ]:
# Load the dataset
full_metadata = metadata.reset_index(drop=True)

X, y = prepare_cnn_data(full_metadata)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (300, 128, 173, 1)
y shape: (300,)


In [ ]:
# Split the data while preserving class proportions in each subset.
num_classes = metadata["classID"].nunique()
y_categorical = to_categorical(y, num_classes=num_classes)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
# A CNN learns local time-frequency patterns from the Mel spectrogram images.
input_shape = X_train.shape[1:]
model = models.Sequential([
    layers.Input(shape=input_shape),
    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(num_classes, activation="softmax")
])

model.summary()

In [ ]:
# Train the network. Early stopping limits overfitting and restores the best weights.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6)
]

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

# Plot training and validation performance.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(history.history["accuracy"], label="Training")
axes[0].plot(history.history["val_accuracy"], label="Validation")
axes[0].set_title("Model accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training")
axes[1].plot(history.history["val_loss"], label="Validation")
axes[1].set_title("Model loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate once on the held-out test set.
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print("Test loss:", round(float(test_loss), 4))
print("Test accuracy:", round(float(test_accuracy), 4))

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
class_names = (
    metadata[["classID", "class"]]
    .drop_duplicates()
    .sort_values("classID")["class"]
    .tolist()
)

print("\nClassification report:")
print(classification_report(
    y_true, y_pred,
    labels=list(range(num_classes)),
    target_names=class_names,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(11, 9))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("Confusion matrix on the test set")
plt.tight_layout()
plt.show()

# Save the trained model for later use.
model.save("urbansound8k_cnn.keras")
print("Saved model to urbansound8k_cnn.keras")

## Notes on interpreting the results

The test accuracy measures the proportion of held-out clips assigned the correct sound class. The classification report adds precision, recall and F1-score for each class, while the confusion matrix shows which sound categories the model most often mixes up.

The split is stratified and reproducible, but it is a random clip-level split. UrbanSound8K contains clips grouped into folds and source recordings, so a stricter follow-up experiment could train on selected folds and evaluate on a completely held-out fold to test generalisation to unseen recordings. Results can also vary with the random seed, hardware, preprocessing choices and training time.